## Automatic video annotation using SAM3 video text-prompt model

In [1]:
import gc
import os
import tempfile
import atexit
import json
import shutil
import sys
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from skimage import measure
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SAM3_DIR = PROJECT_ROOT / "pkg" / "sam3"
if str(SAM3_DIR) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR))

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

gpus_to_use = range(torch.cuda.device_count())
predictor = build_sam3_video_predictor(checkpoint_path="../weights/sam3/sam3.pt",
    gpus_to_use=gpus_to_use)
# Detection confidence threshold: lower = more detections, higher = stricter
predictor.model.score_threshold_detection = 0.8
print("SAM3 video predictor loaded")

/home/kewei/anaconda3/envs/auto_annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/kewei/anaconda3/envs/auto_annotation/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
INFO 2026-06-23 13:01:36,724 70324 sam3_video_predictor.py: 109: using the following GPU IDs: [0]
INFO 2026-06-23 13:01:36,788 70324 sam3_video_predictor.py: 125: 


	*** START loading model on all ranks ***


INFO 2026-06-23 13:01:36,788 70324 sam3_video_predictor.py: 127: loading model on rank=0 with world_size=1 -- this could take a while ...
INFO 2026-06-23 13:01:39,324 70324 sam3_video_base.py: 348: setting 

SAM3 video predictor loaded


### Configure class to text prompt mapping

In [3]:
# class_id -> (Chinese label, English text prompt for SAM3)
class_config = {
    0: ("red box", "red box"),
    1: ("small object", "small object"),
    2: ("green small box", "green small box"),
    3: ("toy car", "toy car"),
}

# Video path: either a folder of JPEG frames or a .mp4 file
video_path = str(PROJECT_ROOT / "datasets/task_video/0001.mp4")
annotation_save_path = PROJECT_ROOT / "datasets/task_video/annotations/annotations.json"
frame_save_dir = PROJECT_ROOT / "datasets/task_video/frames"  # save extracted frames here

prompt_frame_idx = 0  # which frame to apply the text prompt on

# Frame sampling: only annotate every Nth frame to reduce computation
source_fps = 30       # original video FPS (for reference)
target_fps = 2       # desired annotation FPS (set <= source_fps)
frame_stride = max(1, source_fps // target_fps)  # e.g. 30//5=6, annotate every 6th frame

print(f"Video path: {video_path}")
print(f"Frame save dir: {frame_save_dir}")
print(f"Prompt frame index: {prompt_frame_idx}")
print(f"Source FPS={source_fps}, target FPS={target_fps} -> frame_stride={frame_stride}")
print(f"Classes: {[v[0] for v in class_config.values()]}")

Video path: /home/kewei/automatic-annotation/datasets/task_video/0001.mp4
Frame save dir: /home/kewei/automatic-annotation/datasets/task_video/frames
Prompt frame index: 0
Source FPS=30, target FPS=2 -> frame_stride=15
Classes: ['red box', 'small object', 'green small box', 'toy car']


## Run SAM3 video text-prompt inference & save COCO annotations

In [4]:
def propagate_and_collect(predictor, session_id, sampled_set, frame_to_image_id,
                           coco_annotations, class_id, img_w, img_h, ann_id_start, min_area=0):
    """
    Stream propagation results and extract COCO annotations on-the-fly.
    Only keeps sampled frames' outputs; discards others immediately.
    Returns (next_ann_id, count_this_class).
    """
    ann_id = ann_id_start
    obj_count = 0
    inference_state = predictor._all_inference_states[session_id]["state"]
    cached_outputs = inference_state["cached_frame_outputs"]
    for response in predictor.handle_stream_request(
        request=dict(
            type="propagate_in_video",
            session_id=session_id,
        )
    ):
        frame_idx = response["frame_index"]
        outputs = response.get("outputs")
        # ── immediately clear this frame's GPU cache to avoid OOM ──
        cached_outputs.pop(frame_idx, None)
        if frame_idx not in sampled_set or outputs is None:
            continue
        segms = outputs_to_coco_annotations(outputs, img_w, img_h)
        for seg, bbox, area in segms:
            if area < min_area:
                continue
            coco_annotations.append({
                "id": ann_id,
                "image_id": frame_to_image_id[frame_idx],
                "category_id": class_id,
                "segmentation": [seg],
                "bbox": bbox,
                "area": area,
                "iscrowd": 0,
            })
            ann_id += 1
            obj_count += 1
    return ann_id, obj_count


def outputs_to_coco_annotations(outputs, image_w, image_h):
    """
    Convert a single frame's outputs dict to list of (segmentation, bbox_xywh, area).
    outputs keys: out_obj_ids, out_binary_masks, out_boxes_xywh, out_probs
    """
    results = []
    if outputs is None or len(outputs.get("out_obj_ids", [])) == 0:
        return results

    obj_ids = outputs["out_obj_ids"]
    masks = outputs["out_binary_masks"]
    boxes_xywh = outputs["out_boxes_xywh"]

    for i in range(len(obj_ids)):
        mask_np = np.asarray(masks[i])
        if mask_np.ndim == 2:
            mask_np = mask_np
        else:
            mask_np = mask_np.squeeze(0) if mask_np.ndim == 3 else mask_np
        mask_np = mask_np.astype(np.uint8)
        if not mask_np.any():
            continue

        contours = measure.find_contours(mask_np.astype(np.float64), 0.5)
        if not contours:
            continue
        contour = max(contours, key=len)
        contour_xy = np.fliplr(contour)
        segmentation = contour_xy.flatten().tolist()

        # boxes_xywh: [x, y, w, h] in pixel coords
        x, y, w, h = boxes_xywh[i]
        bbox = [float(x), float(y), float(w), float(h)]

        poly = contour_xy
        area = 0.5 * abs(np.dot(poly[:, 0], np.roll(poly[:, 1], 1)) -
                         np.dot(poly[:, 1], np.roll(poly[:, 0], 1)))

        results.append((segmentation, bbox, float(area)))

    return results


def get_video_frame_paths(video_path):
    """Return sorted list of frame file paths, or integer range for mp4."""
    import glob
    if video_path.endswith(".mp4"):
        import cv2
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        return list(range(frame_count))
    else:
        frame_paths = glob.glob(str(Path(video_path) / "*.jpg"))
        try:
            frame_paths.sort(key=lambda p: int(Path(p).stem))
        except ValueError:
            frame_paths.sort()
        return [Path(p) for p in frame_paths]


def get_frame_size(video_path):
    """Get (width, height) of the first frame."""
    import glob
    import cv2
    if video_path.endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        ret, frame = cap.read()
        cap.release()
        if ret:
            return frame.shape[1], frame.shape[0]
    else:
        frame_paths = glob.glob(str(Path(video_path) / "*.jpg"))
        if frame_paths:
            img = Image.open(frame_paths[0])
            return img.size
    return 1920, 1080


def save_frame_image(video_path, frame_idx, frame_save_dir):
    """Save a single frame to frame_save_dir as frame_{idx:05d}.jpg."""
    import cv2
    frame_save_dir = Path(frame_save_dir)
    frame_save_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"frame_{frame_idx:05d}.jpg"
    out_path = frame_save_dir / out_name
    if out_path.exists():
        return out_name  # already saved
    if video_path.endswith(".mp4"):
        cap = cv2.VideoCapture(video_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        cap.release()
        if ret:
            cv2.imwrite(str(out_path), frame)
    else:
        import glob
        frame_paths = sorted(glob.glob(str(Path(video_path) / "*.jpg")))
        if frame_idx < len(frame_paths):
            shutil.copy2(frame_paths[frame_idx], out_path)
    return out_name


def cleanup_gpu():
    """Release GPU memory and run garbage collection."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def make_coco_video_annotation(video_path, class_config, predictor,
                                prompt_frame_idx, save_path,
                                frame_stride=1, frame_save_dir=None, min_area=0):
    """
    Run SAM3 video text-prompt on a video (per-class sessions),
    collect per-frame masks on-the-fly, and append to COCO JSON.
    If save_path already exists, new annotations are merged in.
    """
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    frame_paths = get_video_frame_paths(video_path)
    num_frames = len(frame_paths)
    img_w, img_h = get_frame_size(video_path)

    sampled_frame_idxs = list(range(0, num_frames, frame_stride))
    sampled_set = set(sampled_frame_idxs)

    # ── Load existing COCO JSON if present (append mode) ──
    existing_images = []
    existing_annotations = []
    existing_categories = {}
    existing_frame_to_image_id = {}

    if save_path.exists() and save_path.stat().st_size > 0:
        with open(save_path, "r", encoding="utf-8") as f:
            existing = json.load(f)
        existing_images = existing.get("images", [])
        existing_annotations = existing.get("annotations", [])
        for cat in existing.get("categories", []):
            existing_categories[cat["id"]] = cat["name"]
        for img in existing_images:
            existing_frame_to_image_id[img["frame_idx"]] = img["id"]
        print(f"Loaded existing: {len(existing_images)} images, "
              f"{len(existing_annotations)} annotations, "
              f"{len(existing_categories)} categories")

    # ── Build COCO images (skip already-existing frames) ──
    max_img_id = max((img["id"] for img in existing_images), default=0)
    coco_images = list(existing_images)
    frame_to_image_id = dict(existing_frame_to_image_id)

    new_frame_count = 0
    for frame_idx in tqdm(sampled_frame_idxs, desc="Saving frames"):
        if frame_idx in frame_to_image_id:
            continue  # frame already in existing annotations
        new_frame_count += 1
        max_img_id += 1
        file_name = save_frame_image(video_path, frame_idx, frame_save_dir) if frame_save_dir else f"{frame_idx:05d}.jpg"
        frame_to_image_id[frame_idx] = max_img_id
        coco_images.append({
            "id": max_img_id,
            "file_name": file_name,
            "frame_idx": frame_idx,
            "width": img_w,
            "height": img_h,
        })

    print(f"Video: {num_frames} total frames, {img_w}x{img_h}")
    print(f"Frame stride={frame_stride} -> annotating {len(sampled_frame_idxs)} frames")
    if new_frame_count > 0:
        print(f"  {new_frame_count} new frames, {len(existing_images)} already present")

    # ── Extract only sampled frames to temp dir for SAM3 (saves GPU memory) ──
    # SAM3 loads ALL frames in a directory to GPU; by giving it only sampled frames,
    # GPU memory drops from 2793 frames to {len(sampled_frame_idxs)} frames.
    import tempfile, atexit
    tmp_dir = tempfile.mkdtemp(prefix="sam3_frames_")
    atexit.register(lambda: shutil.rmtree(tmp_dir, ignore_errors=True))
    orig_to_sam3_idx = {}
    for sam3_idx, orig_idx in enumerate(sampled_frame_idxs):
        orig_to_sam3_idx[orig_idx] = sam3_idx
        # Copy frame to tmp dir with sequential name (0.jpg, 1.jpg, ...)
        src_name = f"frame_{orig_idx:05d}.jpg"
        dst_name = f"{sam3_idx}.jpg"
        src = (Path(frame_save_dir) / src_name) if frame_save_dir else None
        dst = Path(tmp_dir) / dst_name
        if frame_save_dir and src.exists():
            shutil.copy2(src, dst)
        else:
            save_frame_image(video_path, orig_idx, tmp_dir)
            saved = Path(tmp_dir) / src_name
            if saved.exists():
                shutil.move(str(saved), str(dst))
    # SAM3 views these as frame 0..N-1; all of them need annotation
    sam3_video_path = str(tmp_dir)
    sam3_sampled_set = set(range(len(sampled_frame_idxs)))
    sam3_frame_to_image_id = {i: frame_to_image_id[sampled_frame_idxs[i]]
                              for i in range(len(sampled_frame_idxs))}
    sam3_prompt_idx = orig_to_sam3_idx.get(prompt_frame_idx, 0)

    # ── Merge categories (skip already-present class_ids) ──
    combined_categories = [
        {"id": cid, "name": name, "supercategory": "object"}
        for cid, name in existing_categories.items()
    ]
    existing_cat_ids = set(existing_categories.keys())
    for cid, (label_, _) in class_config.items():
        if cid not in existing_cat_ids:
            combined_categories.append({
                "id": cid, "name": label_, "supercategory": "object"
            })
        else:
            print(f"  SKIP class [{label_}] (id={cid}): already in existing annotations")

    # ── Per-class sessions ──
    max_ann_id = max((a["id"] for a in existing_annotations), default=0)
    coco_annotations = list(existing_annotations)
    ann_id = max_ann_id + 1
    total_objs = 0
    classes_to_run = {
        cid: v for cid, v in class_config.items() if cid not in existing_cat_ids
    }

    for class_id, (label, text_prompt) in tqdm(classes_to_run.items(), desc="Classes"):
        print(f"Class [{label}]: text prompt = '{text_prompt}'")

        # Start session (offload video to CPU to save GPU memory)
        response = predictor.handle_request(
            request=dict(type="start_session", resource_path=sam3_video_path,
                         offload_video_to_cpu=True)
        )
        session_id = response["session_id"]

        # Add text prompt
        predictor.handle_request(
            request=dict(
                type="add_prompt",
                session_id=session_id,
                frame_index=sam3_prompt_idx,
                text=text_prompt,
            )
        )

        # Propagate + extract annotations on the fly (no full dict buffered)
        ann_id, obj_count = propagate_and_collect(
            predictor, session_id, sam3_sampled_set, sam3_frame_to_image_id,
            coco_annotations, class_id, img_w, img_h, ann_id, min_area)
        total_objs += obj_count
        print(f"  Found {obj_count} instance-frame annotations")

        # Close session + aggressive GPU cleanup
        predictor.handle_request(
            request=dict(type="close_session", session_id=session_id)
        )
        # extra GC passes to ensure all tensor refs are released
        gc.collect()
        gc.collect()
        cleanup_gpu()

    # Clean up temp directory
    shutil.rmtree(tmp_dir, ignore_errors=True)

    # ── Assemble and save COCO JSON ──
    coco_output = {
        "images": coco_images,
        "annotations": coco_annotations,
        "categories": combined_categories,
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    # Save labels.txt (rebuild from merged categories)
    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:
        for cat in sorted(combined_categories, key=lambda c: c["id"]):
            f_labels.write(cat["name"])

    print(f"COCO annotation saved to {save_path}")
    print(f"   images={len(coco_images)}, "
          f"annotations={len(coco_annotations)}, "
          f"categories={len(combined_categories)}")
    print(f"   total objects found (across sampled frames): {total_objs}")

In [ ]:
# Run auto-annotation
annotation_save_path.parent.mkdir(parents=True, exist_ok=True)

make_coco_video_annotation(
    video_path=video_path,
    class_config=class_config,
    predictor=predictor,
    prompt_frame_idx=prompt_frame_idx,
    save_path=annotation_save_path,
    frame_stride=frame_stride,
    frame_save_dir=frame_save_dir,
    min_area=0,
)

Video: 2793 total frames, 640x480
Frame stride=15 -> annotating 187 frames


Classes:   0%|          | 0/4 [00:00<?, ?it/s]

Class [red box]: text prompt = 'red box'



frame loading (OpenCV) [rank=0]: 100%|██████████| 2793/2793 [00:01<00:00, 2477.68it/s]
INFO 2026-06-23 13:01:50,718 70324 sam3_base_predictor.py: 146: started new session 8e592a47-c138-4baf-8d03-e820496b0188

propagate_in_video:  69%|██████▉   | 1924/2793 [03:20<01:36,  9.05it/s]

In [ ]:
# Clean up: shutdown predictor to free GPU resources
predictor.shutdown()